# Machine Learning Dataset Preparation

This notebook prepares a clean supervised-learning dataset for the Vietnamese ETF Decision Support System. It does not train a model and does not write to PostgreSQL.

Target definition: `future_return_5d = close.shift(-5) / close - 1`, calculated separately for each ETF. The binary target is `1` when the future 5-trading-day return is positive and `0` otherwise.

## 1. Setup

Use pandas, numpy, SQLAlchemy, and the shared backend feature-building code. Database credentials are loaded from the existing backend configuration.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_ROOT = PROJECT_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.core.config import settings
from app.ml.features import (
    FEATURE_COLUMNS,
    build_ml_dataset,
    load_ml_source_data,
    purged_chronological_split,
    validate_purged_split,
)

engine = create_engine(settings.database_url, pool_pre_ping=True)
FEATURE_COLUMNS

## 2. Load Source Data

Load joined price history and technical indicator data from PostgreSQL. This is read-only and uses `price_history` joined to `technical_indicators` by ETF and date.

In [ ]:
source_df = load_ml_source_data(engine)
source_df.head()

## 3. Build ML Dataset

The dataset builder calculates features using only information available at time `t`. The only future-looking operation is `close.shift(-5)`, and it is used only to create the target, not as a feature. Rows without enough future data or enough indicator lookback data are removed.

In [ ]:
ml_df = build_ml_dataset(source_df, horizon_days=5)
ml_df.head()

## 4. Dataset Shape and Coverage

Summarize the dataset size, rows per ETF, and date range after removing rows that cannot be used for supervised learning.

In [ ]:
print("Dataset shape:", ml_df.shape)

rows_per_etf = ml_df.groupby("symbol").size().reset_index(name="rows")
date_coverage = ml_df.groupby("symbol").agg(min_date=("date", "min"), max_date=("date", "max")).reset_index()

display(rows_per_etf)
display(date_coverage)

## 5. Target Class Distribution

Check whether the binary target is balanced overall and within each ETF.

In [ ]:
target_distribution = ml_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="rows")
target_distribution["percentage"] = target_distribution["rows"] / target_distribution["rows"].sum()
target_distribution

In [ ]:
target_by_etf = (
    ml_df.groupby(["symbol", "target"])
    .size()
    .reset_index(name="rows")
)
target_by_etf["percentage"] = target_by_etf["rows"] / target_by_etf.groupby("symbol")["rows"].transform("sum")
target_by_etf

## 6. Feature Statistics and Quality Checks

Inspect descriptive statistics, missing values, and infinite values for model input features.

In [ ]:
ml_df[FEATURE_COLUMNS].describe()

In [ ]:
missing_values = ml_df[FEATURE_COLUMNS + ["future_return_5d", "target"]].isna().sum()
infinite_values = np.isinf(ml_df[FEATURE_COLUMNS + ["future_return_5d"]]).sum()

display(missing_values.rename("missing_count"))
display(infinite_values.rename("infinite_count"))

## 7. Feature Correlation

Review correlations between numeric input features. This is diagnostic only; no feature selection or model training is performed here.

In [ ]:
feature_correlation = ml_df[FEATURE_COLUMNS].corr()
feature_correlation

## 8. Purged Chronological Train/Test Split Design

For financial time-series data, the split must be chronological. Random `train_test_split` is inappropriate because it can mix future observations into the training set and create leakage.

This dataset predicts a 5-trading-period future return. A normal chronological split is still not enough: training rows immediately before the test cutoff may use target prices that occur inside the test period. The split below applies one common test start date, then purges training rows whose `target_date` is on or after the test start date. The purge uses ETF-specific trading-observation order through the precomputed `target_date`; it does not subtract calendar days.

In [ ]:
unique_dates = pd.Series(sorted(ml_df["date"].unique()))
cutoff_index = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates.iloc[cutoff_index]

train_df, test_df, purged_df = purged_chronological_split(
    ml_df,
    test_start_date=cutoff_date,
    horizon=5,
)
validate_purged_split(train_df, test_df, test_start_date=cutoff_date)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": len(train_df),
            "min_date": train_df["date"].min(),
            "max_date": train_df["date"].max(),
        },
        {
            "split": "purged",
            "row_count": len(purged_df),
            "min_date": purged_df["date"].min(),
            "max_date": purged_df["date"].max(),
        },
        {
            "split": "test",
            "row_count": len(test_df),
            "min_date": test_df["date"].min(),
            "max_date": test_df["date"].max(),
        },
    ]
)

train_target_distribution = train_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="train_rows")
test_target_distribution = test_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="test_rows")

print("Original cutoff / test start date:", cutoff_date)
display(split_summary)
display(train_target_distribution)
display(test_target_distribution)